# LangChain Toolkit


## OpenAI LLM 준비


In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from sqlalchemy import create_engine, text

# 1. SQLite DB 생성 (메모리 DB)
engine = create_engine("sqlite:///example.db")

# 2. 샘플 테이블 생성 및 데이터 삽입
with engine.connect() as conn:
    conn.execute(
        text(
            """
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            age INTEGER
        )
    """
        )
    )

    conn.execute(
        text(
            """
        INSERT INTO users (name, age) VALUES
        ('Alice', 30),
        ('Bob', 25),
        ('Charlie', 35)
    """
        )
    )
    conn.commit()

# 3. LangChain SQLDatabase 객체 생성
db = SQLDatabase(engine)

# 4. LLM 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 5. Toolkit 생성
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

# 6. Agent 생성 (최신 방식)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 데이터베이스 전문가입니다. SQL을 정확하게 작성하세요.",
)

# 7. 질의 실행
query = "users 테이블에서 나이가 가장 많은 사람은 누구인가요?"

response = agent.invoke({"messages": [{"role": "user", "content": query}]})

print("에이전트 응답:", response["messages"][-1].content)
response["messages"]

에이전트 응답: users 테이블에서 나이가 가장 많은 사람은 Charlie입니다.


[HumanMessage(content='users 테이블에서 나이가 가장 많은 사람은 누구인가요?', additional_kwargs={}, response_metadata={}, id='c89bd5ed-fda9-43f4-a371-52e2c6245174'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 348, 'total_tokens': 360, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4fe3d90af3', 'id': 'chatcmpl-DLPyh6sDWGvix5LKXKtoIVFNPbx0C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0a77-d557-7fc3-9ba0-2ea261bd0059-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {}, 'id': 'call_zny7EgxCOxlyfMU4hC7RcCxn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 348, 'output_tokens': 12, 'total_